# DeBERTa hyperparameter grid — LR {1e-5, 2e-5} × epochs {3, 5}

E04 ran deberta-v3-base at the T9 config (LR 3e-5, 5 epochs) and it beat bert-base
by +0.065 equi. But the per-epoch curves peak right at the end of warmup and dip
hard at epoch 2 while LR is highest — precision up, recall collapsing — which is the
signature of 3e-5 being too high for this encoder. 3e-5 was selected in T9 **on BERT**.

Four new cells, 5 seeds each, 20 runs. `{3e-5, 5}` is E04 and is not recomputed;
`{3e-5, 3}` is absent unless you add it.

**Budget ~2h10m** — 3-epoch runs ~5 min, 5-epoch ~8 min. That is long for one free
Colab session, so each cell is zipped and downloaded the moment it finishes: a
disconnect costs one cell, not the lot.

**Runtime → Change runtime type → T4 GPU.**


In [ ]:
# 1. Colab-only guard, GPU check, and the command helper used below.
try:
    import google.colab  # noqa: F401
except ImportError:
    raise SystemExit(
        'This notebook runs on Google Colab only.\n'
        'Open it at colab.research.google.com -> File -> Upload notebook,\n'
        'or File -> Open notebook -> GitHub -> ahmedwaleedaref/ATE-ACTER.')

import subprocess, sys
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip())
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU.'
print('torch', torch.__version__, '| cuda', torch.version.cuda)

def run(*args):
    """Stream a command into the cell, raise on failure. Plain subprocess:
    a multi-line ! with continuations inside a loop is invalid after IPython's
    transform."""
    import os
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}   # else the child block-buffers
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {" ".join(args)}')


In [ ]:
# 2. Clone the project and the corpus
%cd /content
!rm -rf ate-acter
!git clone -q https://github.com/ahmedwaleedaref/ATE-ACTER.git ate-acter
%cd /content/ate-acter
!git log --oneline -1
!git clone -q https://github.com/AylaRT/ACTER.git data/raw/ACTER
!cd data/raw/ACTER && git checkout -q f05b09e985cad37eeaa8daa8b3f383197aa5324e

import subprocess
log = subprocess.run(['git','log','--oneline','-60'], capture_output=True, text=True).stdout
assert '--model override' in log, 'repo predates --model: push 179a678 first'
assert 'forces fp32'   in log, 'repo predates the fp32 fix: push 57f1b78, or DeBERTa will NaN'
assert __import__('pathlib').Path('data/raw/ACTER/en/htfl/annotated').is_dir(), 'ACTER checkout looks wrong'
print('\nrepo + corpus OK')


In [ ]:
# 3. Pinned installs.
#
# torch/numpy/scikit-learn are deliberately NOT pinned: Colab ships a torch built
# for its own driver, and forcing the local 2.14.0 pulls a mismatched CUDA build.
# The run JSON records the torch version, so the deviation stays in the record.
!pip install transformers==5.16.1 tokenizers==0.23.1 safetensors==0.8.0 \
             huggingface_hub==1.29.0 sentencepiece==0.2.2 \
             protobuf==7.36.0 PyYAML==6.0.3 pytest==8.3.2

# seqeval ships an sdist only, and its legacy setup.py fails `egg_info` under
# recent setuptools. It is NOT on the training path: the sole importer is
# tests/test_seqeval_agreement.py, which cross-checks score_exact_spans against
# seqeval. Best-effort, and the run is unaffected if it will not build.
def _pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a],
                          capture_output=True, text=True).returncode == 0

HAVE_SEQEVAL = (_pip('--no-build-isolation', 'seqeval==1.2.2')
                or (_pip('setuptools<81', 'wheel')
                    and _pip('--no-build-isolation', 'seqeval==1.2.2')))
print('\nseqeval:', 'installed (47 tests)' if HAVE_SEQEVAL else
      'UNAVAILABLE -- 46 of 47 tests will run; training is unaffected')


In [ ]:
# 4. Environment check. A version gap is a T10 confound -- record it in E04.
import importlib
for mod in ('torch', 'transformers', 'tokenizers', 'numpy'):
    print(f'{mod:14} {importlib.import_module(mod).__version__}')
print('python', sys.version.split()[0], ' (local: 3.14.4, torch 2.14.0, transformers 5.16.1)')
print()
skip = [] if HAVE_SEQEVAL else ['--ignore=tests/test_seqeval_agreement.py']
if skip:
    print('NOTE: test_seqeval_agreement.py deselected -- seqeval did not install.')
    print('      score_exact_spans is then unverified against seqeval on this machine.')
run(sys.executable, '-m', 'pytest', 'tests/', '-q', *skip)


In [ ]:
# 5. The grid. Each cell downloads as soon as its five seeds finish.
import json, pathlib, shutil
from google.colab import files

SEEDS = (42, 43, 44, 45, 46)
CELLS = [(lr, ep) for lr in ('1e-5', '2e-5') for ep in (3, 5)]

def summarise(d, seed):
    r = json.loads((d / f'seed_{seed}.json').read_text())
    print(f"    SEED {seed}: best_epoch={r['best_epoch']} equi={r['best_equi_f1']:.4f} "
          f"htfl={r['htfl_f1']:.4f} collapsed={r['collapsed']} {r['wall_time_sec']}s")

for lr, ep in CELLS:
    group = f'deberta_grid/deberta_lr{float(lr):g}_e{ep}'
    d = pathlib.Path('results/runs') / group
    print(f'\n########## LR {lr}, {ep} epochs ##########')
    for s in SEEDS:
        run(sys.executable, '-m', 'src.models.run_train',
            '--model', 'microsoft/deberta-v3-base',
            '--lr', lr, '--epochs', str(ep),
            '--group', group, '--seed', str(s),
            '--reason', f'deberta grid lr={lr} epochs={ep}')
        summarise(d, s)
    # cell complete -- get it off /content before anything can drop
    z = shutil.make_archive(f'/content/deberta_lr{float(lr):g}_e{ep}', 'zip', d)
    files.download(z)
    print(f'downloaded {z}')


In [ ]:
# 6. Optional: {3e-5, 3}, the one cell missing from a full 3 x 2 grid.
#    {3e-5, 5} already exists as E04. ~25 min.
import shutil
group = 'deberta_grid/deberta_lr3e-05_e3'
d = pathlib.Path('results/runs') / group
for s in SEEDS:
    run(sys.executable, '-m', 'src.models.run_train',
        '--model', 'microsoft/deberta-v3-base',
        '--lr', '3e-5', '--epochs', '3',
        '--group', group, '--seed', str(s),
        '--reason', 'deberta grid lr=3e-5 epochs=3')
    summarise(d, s)
z = shutil.make_archive('/content/deberta_lr3e-05_e3', 'zip', d)
files.download(z)


In [ ]:
# 7. Grid table. Cells you have not run show as '—'; E04's cell only resolves
#    once results/runs/t10/deberta-v3-base is present in this checkout, which it
#    is not on Colab -- run this locally after committing everything.
run(sys.executable, '-m', 'src.aggregate', '--grid', 'deberta')
